In [1]:
import spacy, re, nltk
import numpy as np
from textstat import textstat


In [2]:
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")



def initialize_dependencies():
    for res in ['cmudict', 'punkt', 'punkt_tab']:
        nltk.download(res, quiet=True)
    try:
        from nltk.corpus import cmudict
        _ = cmudict.dict()
        print("NLP dependencies initialized")
    except Exception as e:
        print(f"Warning: NLTK pre-load encountered an issue: {e}")
initialize_dependencies()


NLP dependencies initialized


In [3]:
#query classsification 
class QueryClassifier:
    def __init__(self):
        self.patterns = {
            'factual': [r'^(what|who|when|where|which)\s', r'define\s', r'what is\s'], # llm 
            'how_to': [r'^how (to|do|can)', r'steps to', r'guide to'], # lrm > llm
            'reasoning': [r'^why\s', r'explain why', r'reasoning behind'], # lrm
            'creative': [r'write (a|an)', r'create', r'generate', r'imagine'], # llm 
            'comparison': [r'(compare|difference between|versus|vs)', r'better than', r'pros and cons'] # lrm
        }
    
    def classify(self, prompt):
        prompt_lower = prompt.lower()
        scores = {}
        for category, patterns in self.patterns.items():
            score = sum(1 for pattern in patterns if re.search(pattern, prompt_lower))
            if score > 0:
                scores[category] = score
        return max(scores.items(), key=lambda x: x[1])[0] if scores else ' '


In [ ]:
# length analysis using pre-trained model to determine 
def get_complexity_metrics(query):
    doc = nlp(query)
    
    depth = max([len(list(token.ancestors)) for token in doc]) if len(doc) > 0 else 0
    negations = [tok for tok in doc if tok.dep_ == "neg" or tok.lemma_ in ["avoid", "exclude"]]
    logic_ops = [tok for tok in doc if tok.lemma_ in ["if", "then", "unless", "whenever"]]
    
    readability = textstat.flesch_kincaid_grade(query)
    score = (depth * 0.4) + (len(logic_ops) * 0.3) + (len(negations) * 0.3)
    
    return {
        "score": score,
        "depth": depth,
        "logic_count": len(logic_ops),
        "negation_count": len(negations),
        "readability": readability
    }


In [ ]:
# breadth analysis using pre-trained model to determine question domain variance
def ner_analysis(query=""):
    doc = nlp(query)
    try:
        count = {}
        for ent in doc.ents:
            count[ent.label_] = count.get(ent.label_, 0) + 1

        # numerical conversion: domain-crossing only
        IGNORE_TYPES = {"DATE", "TIME", "PERSON"}
        breadth = len([t for t in count if t not in IGNORE_TYPES])
        if (breadth <= 0): return 0.0
        elif (breadth == 1): return 0.25
        elif (breadth == 2): return 0.5
        elif (breadth == 3): return 0.75
        else: return 1.0 # multi-domain  
    except:
        raise ValueError

{'score': 0.8, 'depth': 2, 'logic_count': 0, 'negation_count': 0, 'readability': 0.7199999999999989}


In [ ]:
def route_sge(query):
    # featurized vector router function
    classifier = QueryClassifier()
    intent = classifier.classify(query)
    metrics = get_complexity_metrics(query)   
    breadth_analysis = ner_analysis(query)  
    intent_map = {'reasoning': 0, 'comparison': 1, 'how_to': 2, 'creative': 3, 'factual': 4}

    # now 10 features: 5 intent one-hot + 5 metric features (depth, logic, neg, readability, domain_breadth)
    features = np.zeros(10)
    if intent in intent_map:
        features[intent_map[intent]] = 1.0
    # metric slots indices 5..9
    features[5] = metrics['depth']
    features[6] = metrics['logic_count']
    features[7] = metrics['negation_count']
    features[8] = metrics['readability']
    features[9] = breadth_analysis   #

    w_intent = np.array([4.0, 4.0, 1.0, -3.0, -0.5])
    w_metrics = np.array([0.3, 1.2, 1.6, 0.05, 0.8])

    weights = np.concatenate([w_intent, w_metrics])
    bias = -2.5

    logit = np.dot(features, weights) + bias
    probability = 1.0 / (1.0 + np.exp(-logit))
    decision = "LRM" if probability > 0.5 else "LLM"
    return decision
